# Build Dimension - Airport

In [1]:
from pyspark.sql.functions import *

silver_airports = spark.table("silver_airports")

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql.window import Window

dim_airport = (
    silver_airports
    .withColumn(
        "airport_key",
        row_number().over(
            Window.orderBy("iata_code")
        )
    )
)

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 4, Finished, Available, Finished, False)

In [3]:
dim_airport = dim_airport.select(
    "airport_key",
    "iata_code",
    "airport_name",
    "city",
    "country",
    "latitude",
    "longitude"
)

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 5, Finished, Available, Finished, False)

In [4]:
display(dim_airport.limit(10))

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e825671a-3981-4818-8f11-28b7c1be49cb)

In [5]:
dim_airport.write.mode("overwrite").saveAsTable("dim_airport")

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 7, Finished, Available, Finished, False)

In [6]:
flights = spark.table("silver_flights")
airports = spark.table("dim_airport")

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 8, Finished, Available, Finished, False)

In [7]:
airport_match = (
    flights.alias("f")
    .join(
        airports.alias("a"),
        col("f.ORIGIN") == col("a.iata_code"),
        "left"
    )
)

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 9, Finished, Available, Finished, False)

In [8]:
airport_match.filter(
    col("airport_key").isNull()
).count()

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 10, Finished, Available, Finished, False)

149

# Build dim_airline

In [9]:
silver_airlines = spark.table("silver_airlines")

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 11, Finished, Available, Finished, False)

In [10]:
dim_airline = (
    silver_airlines
    .withColumn(
        "airline_key",
        row_number().over(
            Window.orderBy("iata_code")
        )
    )
)

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 12, Finished, Available, Finished, False)

In [11]:
dim_airline = dim_airline.select(
    "airline_key",
    "iata_code",
    "airline_name",
    "country"
)

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 13, Finished, Available, Finished, False)

In [12]:
dim_airline.write.mode("overwrite").saveAsTable("dim_airline")

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 14, Finished, Available, Finished, False)

In [13]:
airport_match.filter(
    col("airport_key").isNull()
).select(
    "ORIGIN"
).distinct().show(50,False)

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 15, Finished, Available, Finished, False)

+------+
|ORIGIN|
+------+
|XWA   |
+------+



# Create Date Dimension

In [14]:
flights = spark.table("silver_flights")

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 16, Finished, Available, Finished, False)

In [15]:
from pyspark.sql.functions import *

dim_date = (
    flights
    .select("flight_date")
    .distinct()
)

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 17, Finished, Available, Finished, False)

In [16]:
dim_date = (
    dim_date
    .withColumn("year", year("flight_date"))
    .withColumn("quarter", quarter("flight_date"))
    .withColumn("month", month("flight_date"))
    .withColumn("month_name", date_format("flight_date","MMMM"))
    .withColumn("day", dayofmonth("flight_date"))
    .withColumn("day_name", date_format("flight_date","EEEE"))
)

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 18, Finished, Available, Finished, False)

In [17]:
from pyspark.sql.window import Window

dim_date = (
    dim_date
    .withColumn(
        "date_key",
        row_number().over(
            Window.orderBy("flight_date")
        )
    )
)

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 19, Finished, Available, Finished, False)

In [18]:
dim_date.write \
.mode("overwrite") \
.saveAsTable("dim_date")

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 20, Finished, Available, Finished, False)

# Build dim_route

In [19]:
flights = spark.table("silver_flights")

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 21, Finished, Available, Finished, False)

In [20]:
dim_route = (
    flights
    .select(
        col("ORIGIN").alias("origin_airport"),
        col("DEST").alias("destination_airport")
    )
    .distinct()
)

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 22, Finished, Available, Finished, False)

In [21]:
dim_route = (
    dim_route
    .withColumn(
        "route_key",
        row_number().over(
            Window.orderBy(
                "origin_airport",
                "destination_airport"
            )
        )
    )
)

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 23, Finished, Available, Finished, False)

In [22]:
dim_route.write \
.mode("overwrite") \
.saveAsTable("dim_route")

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 24, Finished, Available, Finished, False)

In [23]:
print("dim_airport:", spark.table("dim_airport").count())

print("dim_airline:", spark.table("dim_airline").count())

print("dim_date:", spark.table("dim_date").count())

print("dim_route:", spark.table("dim_route").count())

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 25, Finished, Available, Finished, False)

dim_airport: 6072
dim_airline: 1536
dim_date: 31
dim_route: 5666


In [24]:
flights = spark.table("silver_flights")
airlines = spark.table("dim_airline")

airline_match = (
    flights.alias("f")
    .join(
        airlines.alias("a"),
        col("f.OP_UNIQUE_CARRIER") == col("a.iata_code"),
        "left"
    )
)

airline_match.filter(
    col("airline_key").isNull()
).count()

StatementMeta(, bd82efbd-f6e4-4ff1-b496-9e17fd68e323, 26, Finished, Available, Finished, False)

0